In [ ]:
!pip install -q openai datasets pydantic

In [2]:
import json
import os
import re
import textwrap
import time
import warnings
from collections import Counter
from collections.abc import Callable
from concurrent.futures import ThreadPoolExecutor
from statistics import mean
from typing import Literal

from datasets import load_dataset
from openai import OpenAI
from pydantic import BaseModel, ValidationError

warnings.filterwarnings("ignore")

os.environ["OPENAI_API_KEY"] = ""

client = OpenAI()
MODEL = "gpt-5-mini"
INSTRUCT_MODEL = "gpt-4.1-mini"
SEED = 0

# 1) LLM Prompting

A prompt is the program an LLM application runs. Two prompts that look equally reasonable can
differ by tens of accuracy points on the same model, and a prompt tuned for one model can hurt
another. The common failure is judging a prompt on two or three hand-picked examples.

The model type matters as much as the wording. **Instruction models** (`gpt-4.1-mini`) answer
directly unless the prompt asks them to reason. **Reasoning models** (`gpt-5-mini`, o-series,
DeepSeek-R1, Qwen3 in thinking mode) are trained with reinforcement learning to reason in a
hidden channel first, and expose a compute knob (`reasoning_effort`) instead of a magic phrase.

In [3]:
def chat(
    messages: list[dict],
    model: str = INSTRUCT_MODEL,
    effort: str = "minimal",
    **kwargs,
) -> tuple[str, int]:
    """One chat completion with retries; returns (text, output tokens)."""
    if model.startswith("gpt-5"):
        kwargs["reasoning_effort"] = effort
    else:
        kwargs.setdefault("temperature", 0)
    for attempt in range(4):
        try:
            response = client.chat.completions.create(
                model=model, messages=messages, **kwargs
            )
            break
        except Exception:
            if attempt == 3:
                raise
            time.sleep(2**attempt)
    return response.choices[0].message.content, response.usage.completion_tokens


def parallel(fn: Callable, items: list) -> list:
    """Map fn over items with 8 threads, preserving order."""
    with ThreadPoolExecutor(8) as pool:
        return list(pool.map(fn, items))


# Quick test of the chat function
print(chat([{"role": "user", "content": "Reply with the word ready."}]))

('ready', 1)


# 2) Implementation

## 2.1 System-prompt patterns

The system message is read before any user message and most models give it higher priority.
Three patterns cover most production prompts.

**Persona.** A domain-expert identity shifts vocabulary, rigour and how uncertainty is stated.

In [4]:
# Define system persona for the assistant
persona_system = (
    "You are a senior quantitative researcher at a top hedge fund. "
    "You communicate precisely, use mathematical notation where helpful, "
    "and always cite statistical assumptions. "
    "When uncertain, you state the confidence level of your answer."
)
# Build messages including the user prompt
messages = [
    {"role": "system", "content": persona_system},
    {"role": "user", "content": "Explain momentum factor investing in two paragraphs."},
]
print(chat(messages)[0])

Momentum factor investing is a quantitative strategy that exploits the empirical tendency of assets that have performed well in the recent past to continue outperforming in the near future, and conversely for poorly performing assets. Formally, momentum is often measured as the cumulative return over a lookback period \( T \), typically 3 to 12 months, excluding the most recent month to avoid short-term reversal effects. The strategy involves ranking assets by their past returns \( R_{t-T:t-1} \) and taking long positions in the top quantile (winners) and short positions in the bottom quantile (losers). This approach is grounded in behavioral finance theories such as investor underreaction to news and herding behavior, as well as risk-based explanations where momentum captures compensation for bearing certain systematic risks.

Empirically, momentum has been shown to generate statistically significant abnormal returns across various asset classes and geographies, with average monthly a

**Task instructions with delimiters.** Numbered steps, an explicit output format and tags that
mark where the input starts and ends. The tags also stop the model from treating text inside the
input as instructions.

In [5]:
# Define medical coding task instructions
task_system = textwrap.dedent("""\
    You are a medical coding assistant.

    Instructions:
    1. Read the clinical note enclosed in <note>...</note>.
    2. Extract all diagnoses and map each to the most specific ICD-10 code.
    3. Return ONLY a JSON array of objects: [{"diagnosis": ..., "icd10": ...}].
    4. If no diagnosis is found, return an empty array [].
    5. Do NOT add commentary outside the JSON array.""")
note = (
    "<note>Patient presents with acute exacerbation of COPD and "
    "type 2 diabetes mellitus, uncontrolled. BMI 34.</note>"
)
# Create chat messages with the clinical note
messages = [
    {"role": "system", "content": task_system},
    {"role": "user", "content": note},
]
print(chat(messages)[0])

[{"diagnosis": "Acute exacerbation of COPD", "icd10": "J44.1"}, {"diagnosis": "Type 2 diabetes mellitus, uncontrolled", "icd10": "E11.65"}, {"diagnosis": "Obesity, BMI 34", "icd10": "E66.9"}]


**Embedded constraints.** Positive rules (answer from the context) plus negative rules (no
speculation, no dosage advice) and a fixed fallback string. The fallback matters: without it the
model fills gaps with plausible text.

In [6]:
# Define constrained factual Q&A system rules
constrained_system = textwrap.dedent("""\
    You are a factual Q&A assistant for a pharmaceutical company.

    Rules:
    - Answer ONLY based on the provided context.
    - Do NOT speculate or use external knowledge.
    - Do NOT provide dosage recommendations.
    - If the answer is not in the context, reply: "Insufficient information." """)
context = (
    "Context: Drug X showed a 23% reduction in LDL cholesterol in a Phase III "
    "trial (n=4200, p<0.001) over 12 weeks. Common side effects included "
    "headache (8%) and nausea (5%)."
)
# Query the system with example questions and print replies
for question in ("What was the sample size?", "What dose should I take?"):
    messages = [
        {"role": "system", "content": constrained_system},
        {"role": "user", "content": f"{context}\n\nQuestion: {question}"},
    ]
    print(f"{question} -> {chat(messages)[0]}")

What was the sample size? -> The sample size was 4200.


What dose should I take? -> Insufficient information.


## 2.2 Zero-shot vs few-shot

The `dair-ai/emotion` dataset labels tweets with six emotions, several of them easy to confuse
(*joy* vs *love*, *fear* vs *surprise*). 

In [7]:
# Load emotion dataset and prepare train/test splits
emotion = load_dataset("dair-ai/emotion")
LABELS = emotion["train"].features["label"].names
test_items = emotion["test"].shuffle(seed=SEED).select(range(60))
train = emotion["train"].shuffle(seed=SEED)
print(LABELS)
print(Counter(LABELS[row["label"]] for row in test_items))


def demonstrations(per_label: int) -> list[dict]:
    """Balanced demonstrations as alternating user/assistant messages."""
    picked, counts = [], Counter()
    for row in train:
        if counts[row["label"]] < per_label:
            picked.append(row)
            counts[row["label"]] += 1
        if len(picked) == per_label * len(LABELS):
            break
    messages = []
    for row in picked:
        messages += [
            {"role": "user", "content": row["text"]},
            {"role": "assistant", "content": LABELS[row["label"]]},
        ]
    return messages


# Print number of demonstrations produced
print(len(demonstrations(1)) // 2, "demonstrations at 1 per label")

['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
Counter({'sadness': 18, 'joy': 18, 'anger': 9, 'fear': 9, 'love': 5, 'surprise': 1})
6 demonstrations at 1 per label


In [8]:
CLASSIFY_SYSTEM = (
    "Classify the emotion of the text as one of: " + ", ".join(LABELS) + ". "
    "Reply with the label only."
)


def classify_accuracy(per_label: int, model: str = INSTRUCT_MODEL) -> float:
    """Accuracy on the test slice with per_label demonstrations per class."""
    prefix = [{"role": "system", "content": CLASSIFY_SYSTEM}] + demonstrations(
        per_label
    )

    def predict(row: dict) -> bool:
        """True when the predicted label matches the gold label."""
        reply, _ = chat(prefix + [{"role": "user", "content": row["text"]}], model)
        return reply.strip().lower() == LABELS[row["label"]]

    return mean(parallel(predict, list(test_items)))


# Compute few-shot and zero-shot classification accuracies
ICL_RESULTS = {
    f"{INSTRUCT_MODEL}, {k}-shot": classify_accuracy(k // len(LABELS))
    for k in (0, 6, 18)
}
ICL_RESULTS[f"{MODEL}, 0-shot"] = classify_accuracy(0, MODEL)
for config, accuracy in ICL_RESULTS.items():
    print(f"{config:<26} accuracy {accuracy:.2f}")

gpt-4.1-mini, 0-shot       accuracy 0.50
gpt-4.1-mini, 6-shot       accuracy 0.48
gpt-4.1-mini, 18-shot      accuracy 0.48
gpt-5-mini, 0-shot         accuracy 0.45


## 2.3 Chain-of-thought vs reasoning effort

GSM8K grade-school maths problems have a single numeric answer. 

The instruction model is asked
three ways: answer only, "think step by step" (zero-shot CoT), and with two worked examples
(few-shot CoT). The reasoning model is asked for the answer only, at three effort levels.

In [9]:
# Load GSM8K dataset and select problems and exemplars
gsm8k = load_dataset("openai/gsm8k", "main")
problems = gsm8k["test"].shuffle(seed=SEED).select(range(30))
exemplars = gsm8k["train"].select(range(2))


def gold(row: dict) -> float:
    """Gold answer: the number after '####'."""
    return float(row["answer"].split("####")[-1].replace(",", "").strip())


def last_number(text: str) -> float | None:
    """The last number in a reply, the convention for GSM8K scoring."""
    numbers = re.findall(r"-?\d[\d,]*\.?\d*", text or "")
    return float(numbers[-1].replace(",", "").rstrip(".")) if numbers else None


# Build few-shot chain-of-thought exemplars
ANSWER_ONLY = "Reply with only the final number."
STEP_BY_STEP = "Let's think step by step, then give the final number on the last line."
few_shot_cot = []
for row in exemplars:
    worked = re.sub(r"<<.*?>>", "", row["answer"]).replace("####", "Final answer:")
    few_shot_cot += [
        {"role": "user", "content": row["question"]},
        {"role": "assistant", "content": worked},
    ]
print(few_shot_cot[1]["content"])

Natalia sold 48/2 = 24 clips in May.
Natalia sold 48+24 = 72 clips altogether in April and May.
Final answer: 72


In [10]:
def run_gsm8k(
    model: str, prefix: list[dict], suffix: str, effort: str = "minimal"
) -> dict:
    """Accuracy and mean output tokens on the GSM8K slice for one configuration."""

    def solve(row: dict) -> tuple[bool, int]:
        """Answer one problem; return (correct, output tokens)."""
        messages = prefix + [
            {"role": "user", "content": f"{row['question']}\n{suffix}"}
        ]
        reply, tokens = chat(messages, model, effort)
        return last_number(reply) == gold(row), tokens

    rows = parallel(solve, list(problems))
    return {"accuracy": mean(r[0] for r in rows), "tokens": mean(r[1] for r in rows)}


# Run GSM8K configurations and collect results
COT_RESULTS = {
    "4.1-mini answer only": run_gsm8k(INSTRUCT_MODEL, [], ANSWER_ONLY),
    "4.1-mini zero-shot CoT": run_gsm8k(INSTRUCT_MODEL, [], STEP_BY_STEP),
    "4.1-mini few-shot CoT": run_gsm8k(INSTRUCT_MODEL, few_shot_cot, ""),
}
for effort in ("minimal", "low", "medium"):
    COT_RESULTS[f"5-mini effort={effort}"] = run_gsm8k(MODEL, [], ANSWER_ONLY, effort)
print(f"{'configuration':<26}{'accuracy':>9}{'output tokens':>15}")
for config, row in COT_RESULTS.items():
    print(f"{config:<26}{row['accuracy']:>9.2f}{row['tokens']:>15.0f}")

configuration              accuracy  output tokens
4.1-mini answer only           0.40              1
4.1-mini zero-shot CoT         0.93            181
4.1-mini few-shot CoT          0.87            152
5-mini effort=minimal          0.33             10
5-mini effort=low              0.97             95
5-mini effort=medium           0.97            153


## 2.4 Structured output

Extraction into a typed record from AG News articles: a category from a fixed set, the named
entities, and a one-sentence summary. The record is validated by Pydantic, as an application
would before writing it anywhere. Three ways to ask for it:

| Mode | Guarantee |
|---|---|
| Prompt only: "return JSON with these keys" | None; the model may add prose or code fences |
| JSON mode (`response_format={"type": "json_object"}`) | Output parses as JSON |
| Strict schema (`client.chat.completions.parse` with the Pydantic model) | Output parses **and** matches the schema |

Expect prompt-only prompts to look acceptable on small samples but produce occasional parse failures at scale. Use strict response schemas to eliminate parsing errors, and measure content accuracy separately since schemas do not prevent incorrect values.

In [11]:
class NewsRecord(BaseModel):
    """Typed record extracted from one news article."""

    category: Literal["World", "Sports", "Business", "Sci/Tech"]
    entities: list[str]
    summary: str


# Load news test set and define extraction prompt
news = (
    load_dataset("fancyzhx/ag_news", split="test").shuffle(seed=SEED).select(range(20))
)
NEWS_LABELS = ["World", "Sports", "Business", "Sci/Tech"]
EXTRACT_PROMPT = (
    "Extract a record from the article: category (one of World, Sports, Business, "
    "Sci/Tech), entities (named people, organisations and places), and a one-sentence "
    "summary. Return JSON with the keys category, entities, summary."
)


def extract(row: dict, mode: str) -> tuple[bool, bool]:
    """Extract one record in the given mode; return (valid, category correct)."""
    messages = [
        {"role": "system", "content": EXTRACT_PROMPT},
        {"role": "user", "content": row["text"]},
    ]
    if mode == "strict schema":
        response = client.chat.completions.parse(
            model=INSTRUCT_MODEL, messages=messages, response_format=NewsRecord
        )
        record = response.choices[0].message.parsed
        return True, record.category == NEWS_LABELS[row["label"]]
    kwargs = {"response_format": {"type": "json_object"}} if mode == "JSON mode" else {}
    reply, _ = chat(messages, **kwargs)
    try:
        record = NewsRecord.model_validate_json(reply)
    except ValidationError:
        return False, False
    return True, record.category == NEWS_LABELS[row["label"]]

In [12]:
STRUCTURED_RESULTS = {}
# Evaluate extraction modes across the news articles
for mode in ("prompt only", "JSON mode", "strict schema"):
    rows = parallel(lambda row: extract(row, mode), list(news))
    STRUCTURED_RESULTS[mode] = {
        "valid": mean(r[0] for r in rows),
        "category accuracy": mean(r[1] for r in rows),
    }
    print(f"{mode:<14}", {k: round(v, 2) for k, v in STRUCTURED_RESULTS[mode].items()})

# Parse one article using strict schema and print JSON
response = client.chat.completions.parse(
    model=INSTRUCT_MODEL,
    messages=[
        {"role": "system", "content": EXTRACT_PROMPT},
        {"role": "user", "content": news[0]["text"]},
    ],
    response_format=NewsRecord,
)
print(response.choices[0].message.parsed.model_dump_json(indent=1))

prompt only    {'valid': 0.95, 'category accuracy': 0.8}


JSON mode      {'valid': 1, 'category accuracy': 0.85}


strict schema  {'valid': 1, 'category accuracy': 0.85}


{
 "category": "Business",
 "entities": [
  "McTeer",
  "Lonesome Dove",
  "Aggie",
  "CNN/Money",
  "Federal Reserve"
 ],
 "summary": "McTeer, a long-time member of the Federal Reserve system and advocate for the New Economy, is recognized as a fan of Texas picker poets and associated with Lonesome Dove and Aggie."
}


# Summary

| Technique | When it helps | Cost |
|---|---|---|
| System-prompt patterns | Always: role, steps, format, constraints, fallback | Prompt tokens (cacheable) |
| Few-shot demonstrations | Ambiguous labels, unusual formats | Prompt tokens (cacheable) |
| Chain-of-thought prompt | Multi-step problems on instruction models | Visible output tokens |
| Reasoning effort | Multi-step problems on reasoning models | Hidden output tokens |
| Strict schema decoding | Any output a program will parse | None at runtime |